This notebook is to prob (currently with the setting: CGNN-3D, rmsd_cutoff_2, random-k-fold) (linear_probes) for the affinity value and the docking score, and set one as the skyline that would specify the limitation that the embeddings and the linear prob are forcing. <br>

The pipeline to load the $X$ is the same. To make the $y$, one should read the idents of X, load the raw data and sort docking scores/affinities according to the idents from $X$'s idents.

In [ ]:
from pathlib import Path
import os
import pandas as pd

from kinodata.data import KinodataDockedAgnostic

from prob.paths_and_io import get_project_root, get_exp_dirs, load_X_from_pt, load_out_tensor, save_out_tensor
from prob.prob_config import get_ds_load_config
from prob.prob_models import LINEAR_PROBES
from prob.prob_run import run_cv_search



In [4]:
prob_config = get_ds_load_config()

output_dir = prob_config['output_dir']
output_dir

PosixPath('/home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/rmsd_cutoff_2/random-k-fold')

In [5]:
SKYLINE_DIR = output_dir / "general_skylines"
IDS_FILE = "ids.pt"
LAYER_NUM = 3

# Runtime knobs
RANDOM_STATE = 96
N_SPLITS_CV = 5
TEST_SIZE = 0.1


In [6]:
ids = load_out_tensor(output_dir, IDS_FILE)
print(ids.shape, ids.dtype)

torch.Size([41238]) torch.int64


In [3]:
org_ds = KinodataDockedAgnostic()
org_ds_df = org_ds.data_frame
org_ds_df.head()

Loading raw data from /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/raw...
Reading data frame from /home/fatemeh/thesis/kinodata-3D-affinity-prediction/data/raw/kinodata_docked_v2.sdf.gz...
Deduping data frame (current size: 140977)...
138286 complexes remain after deduplication.
Checking for missing pocket mol2 files...


100%|██████████| 3551/3551 [00:00<00:00, 26499.89it/s]


Adding pocket sequences...
(138286, 25)


100%|██████████| 138286/138286 [00:00<00:00, 4115380.68it/s]


Exiting with 3552 cached sequences.
(138286, 26)
Converting to data list...
Done!


,docking.posit_probability,docking.chemgauss_score,activities.activity_id,assays.chembl_id,target_dictionary.chembl_id,molecule_dictionary.chembl_id,molecule_dictionary.max_phase,activities.standard_type,activities.standard_units,compound_structures.canonical_smiles,...,UniprotID,similar.klifs_structure_id,similar.fp_similarity,ID,activities.standard_value,docking.predicted_rmsd,molecule,pocket_mol2_file,ident,structure.pocket_sequence
0,0.18,-13.526784,32335,CHEMBL817617,CHEMBL279,CHEMBL69638,nan,pIC50,nM,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,...,P35968,5326,0.159664,LIG,5.148742,4.720892,<rdkit.Chem.rdchem.Mol object at 0x7fbd5870fd10>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,0,KPLGRGAFGQVIEVAVKMLALMSELKILIHIGLNVVNLLGAMVIVE...
1,0.24,-10.307055,32336,CHEMBL847682,CHEMBL4128,CHEMBL69638,nan,pIC50,nM,Nc1ncnc2c1c(-c1cccc(Oc3ccccc3)c1)cn2C1CCCC1,...,Q02763,5553,0.2,32336,5.468521,5.696663,<rdkit.Chem.rdchem.Mol object at 0x7fbd5870fbc0>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,1,DVIGEG__GQVLKAAIKRM____ELEVLCKLGPNIINLLGAYLAIE...
2,0.18,-11.764866,32680,CHEMBL677833,CHEMBL203,CHEMBL137635,nan,pIC50,nM,CN(c1ccccc1)c1ncnc2ccc(N/N=N/Cc3ccccn3)cc12,...,P00533,12838,0.212329,LIG,5.031517,4.851336,<rdkit.Chem.rdchem.Mol object at 0x7fbd5870ff40>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,2,KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLIMQ...
3,0.24,-10.21195,32770,CHEMBL674643,CHEMBL203,CHEMBL306988,nan,pIC50,nM,CC(=C(C#N)C#N)c1ccc(NC(=O)CCC(=O)[O-])cc1,...,P00533,786,0.148936,LIG,3.301030,6.134686,<rdkit.Chem.rdchem.Mol object at 0x7fbd585d00b0>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,3,KVLGSGAFGTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLITQ...
4,0.18,-3.132142,32773,CHEMBL675636,CHEMBL203,CHEMBL66879,nan,pKi,nM,O=C([O-])/C=C/c1ccc(O)cc1,...,P00533,15067,0.132075,LIG,3.000000,6.192890,<rdkit.Chem.rdchem.Mol object at 0x7fbd585d0120>,/home/fatemeh/thesis/kinodata-3D-affinity-pred...,4,KVLGS___GTVYKVAIKELEILDEAYVMASVDPHVCRLLGIQLIMQ...


In [2]:
full_target = org_ds_df[['ident', 'docking.chemgauss_score', 'activities.standard_value']].rename(columns={'docking.chemgauss_score': 'docking_score', 'activities.standard_value': 'affinity'})
full_target = full_target.set_index('ident')
full_target.head()

NameError: name 'org_ds_df' is not defined

In [1]:
full_target.to_dict()

NameError: name 'full_target' is not defined

In [ ]:
# save the target, similar to other targets as .pt df based on the ids tensor in skyline dir for later use in skyline probing
for col in full_target.columns:
    full_target[col].to_dict()